# Challenge AI Engineer

## Daftar Isi

- **Soal 1B**
- **Soal 2**
- **Soal 3**

## Highlights

- Chatbot FAQ berbasis CLI/Terminal menggunakan model dari Ollama Cloud.
- Menambahkan fitur pemilihan model sehingga pengguna dapat memilih model cloud yang ingin digunakan.
- Topik FAQ: Piala Dunia FIFA 2026

## Soal 1B


### Alur Implementasi

Notebook ini membangun chatbot FAQ sederhana dengan langkah berikut:

1. Menginstal dan mengimpor library yang dibutuhkan.
2. Mengambil daftar model dari Ollama Cloud.
3. Memuat data FAQ dari `faq.txt`.
4. Mencocokkan pertanyaan pengguna dengan FAQ terdekat.
5. Menyediakan perintah `/model`, `/history`, dan `/reset` saat chatbot berjalan.


In [1]:
!pip -q install ollama



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re
from pathlib import Path

import requests
from ollama import Client
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('Library siap digunakan.')


Library siap digunakan.


In [3]:
OLLAMA_API_KEY = "f6a763daaa0546cea30cfdb4ff9bb474.LKi0Qyrj14T0U7mvye2Ad9Ir"
OLLAMA_HEADERS = {"Authorization": f"Bearer {OLLAMA_API_KEY}"}
client = Client(host='https://ollama.com', headers=OLLAMA_HEADERS)

### Pemilihan Model

Bagian ini mengambil daftar model yang tersedia dari Ollama Cloud. Model default tetap `gpt-oss:120b`, lalu perintah `/model` dipakai untuk mengganti model secara manual jika diperlukan.


In [4]:
def model_tags():
    r = requests.get('https://ollama.com/api/tags', headers=OLLAMA_HEADERS, timeout=30)
    r.raise_for_status()
    return [m.get('name') for m in r.json().get('models', []) if m.get('name')]


available_models = model_tags()
if not available_models:
    raise RuntimeError('Tidak ada model cloud yang tersedia.')

default_model = 'gpt-oss:120b'
active_model = default_model
print(f'Model default: {active_model}')


def switch_model():
    global active_model
    print('Model tersedia:')
    for name in available_models:
        print(f'- {name}')
    model = input('Ketik nama model baru: ').strip()
    if model in available_models:
        active_model = model
    print(f'Model aktif sekarang: {active_model}')


Model default: gpt-oss:120b


### Logika FAQ

Pertanyaan pengguna dicocokkan ke FAQ terdekat dengan indeks TF-IDF ringan atas pasangan pertanyaan dan jawaban. Jika skor cocokannya rendah, chatbot memakai jawaban fallback agar tetap konsisten dengan batasan FAQ.


In [5]:
faq_path = Path('faq.txt')
def load_faq(path='faq.txt'):
    pairs = []
    current_q = None
    current_a = None
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line.startswith('Q:'):
            current_q = line[2:].strip()
        elif line.startswith('A:'):
            current_a = line[2:].strip()
        if current_q and current_a:
            pairs.append((current_q, current_a))
            current_q = None
            current_a = None
    return pairs


def recent_context():
    parts = []
    for item in conversation[-MAX_CONTEXT_MESSAGES:]:
        parts.append(item['content'])
    return ' '.join(parts)


faq_vectorizer = None
faq_matrix = None


STOPWORDS = {
    'apa', 'siapa', 'kapan', 'di', 'ke', 'dari', 'dan', 'atau', 'yang', 'untuk',
    'itu', 'ini', 'mana', 'bagaimana', 'kenapa', 'mengapa', 'adalah', 'dengan',
    'pada', 'dalam', 'sebagai', 'terhadap',
}


def build_faq_index(faq_pairs):
    texts = [f'{question} {answer}' for question, answer in faq_pairs]
    vectorizer = TfidfVectorizer(lowercase=True, stop_words=list(STOPWORDS), ngram_range=(1, 2))
    matrix = vectorizer.fit_transform(texts)
    return vectorizer, matrix


def best_faq_match(question, faq_pairs):
    if not faq_pairs or faq_vectorizer is None or faq_matrix is None:
        return (0.0, '', '')
    query_vector = faq_vectorizer.transform([question])
    scores = cosine_similarity(query_vector, faq_matrix)[0]
    best_index = int(scores.argmax())
    return float(scores[best_index]), faq_pairs[best_index][0], faq_pairs[best_index][1]


faq_pairs = load_faq()
faq_vectorizer, faq_matrix = build_faq_index(faq_pairs)
print(f'FAQ loaded: {len(faq_pairs)} item')


FAQ loaded: 30 item


In [6]:
FALLBACK_ANSWER = 'Maaf, saya tidak dapat membantu dengan pertanyaan itu.'
MAX_CONTEXT_MESSAGES = 8

SYSTEM_PROMPT = (
    'Anda adalah chatbot FAQ Piala Dunia FIFA 2026. '
    'Jawab hanya berdasarkan konteks FAQ yang diberikan dan riwayat percakapan yang relevan. '
    'Jangan menambahkan fakta baru, jangan berasumsi, dan jangan menjawab di luar konteks. '
    f'Jika konteks tidak cukup, jawab persis: {FALLBACK_ANSWER}'
)

conversation = []


def build_messages(user_question, matched_question, matched_answer):
    faq_context = f'Konteks FAQ:\nQ: {matched_question}\nA: {matched_answer}\n'
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT + '\n\n' + faq_context}]
    messages.extend(conversation[-MAX_CONTEXT_MESSAGES:])
    messages.append({'role': 'user', 'content': user_question})
    return messages


def ask_model(user_question):
    query = recent_context() + " " + user_question
    score, matched_question, matched_answer = best_faq_match(query, faq_pairs)
    if score < 0.18:
        return FALLBACK_ANSWER

    try:
        response = client.chat(
            model=active_model,
            messages=build_messages(user_question, matched_question, matched_answer),
            stream=False,
        )
        text = response.get('message', {}).get('content', '').strip()
        return text or matched_answer
    except Exception as exc:
        if is_subscription_error(exc):
            print(f'\n[Model {active_model} tidak tersedia karena subscription. Silakan ganti model.]')
            return FALLBACK_ANSWER
        print(f'\n[Fallback karena error Ollama Cloud: {exc}]')
        return matched_answer


def show_history():
    if not conversation:
        print('Belum ada riwayat percakapan.')
        return
    print('\nRiwayat konteks percakapan terakhir:')
    for item in conversation[-MAX_CONTEXT_MESSAGES:]:
        print(f"- {item['role']}: {item['content']}")


def choose_model(prompt, current_model):
    print('Model tersedia:')
    print(f'- {default_model} (default)')
    for name in available_models:
        print(f'- {name}')
    model = input(prompt).strip()
    if model == default_model or model in available_models:
        return model
    return current_model


def is_subscription_error(exc):
    message = str(exc).lower()
    return 'subscription' in message or 'upgrade for access' in message or '403' in message


def switch_model():
    global active_model
    active_model = choose_model('Ketik nama model baru: ', active_model)
    print(f'Model aktif sekarang: {active_model}')


def run_chatbot():
    print(f'Model: {active_model}')
    print('Ketik /model, /history, /reset, exit, atau quit.')

    while True:
        user_input = input('\nUser: ').strip()
        if not user_input:
            print('Bot: Silakan masukkan pertanyaan.')
            continue

        lowered = user_input.lower()
        if lowered in {'exit', 'quit'}:
            print('Bot: Terima kasih. Sesi chatbot selesai.')
            break
        if lowered == '/model':
            switch_model()
            continue
        if lowered == '/history':
            show_history()
            continue
        if lowered == '/reset':
            conversation.clear()
            print('Bot: Riwayat percakapan telah dihapus.')
            continue

        print('Bot: ', end='', flush=True)
        response = ask_model(user_input)
        print(response)
        conversation.append({'role': 'user', 'content': user_input})
        conversation.append({'role': 'assistant', 'content': response})


run_chatbot()



Chatbot FAQ siap digunakan.
Model aktif: gpt-oss:120b
Ketik /model, /history, /reset, exit, atau quit.


Model tersedia:
- gpt-oss:120b (default)
- kimi-k2.6
- qwen3-next:80b
- qwen3-coder-next
- qwen3-vl:235b
- ministral-3:14b
- devstral-2:123b
- gemini-3-flash-preview
- gemma3:4b
- cogito-2.1:671b
- kimi-k2:1t
- kimi-k2-thinking
- qwen3-coder:480b
- minimax-m3
- gemma3:12b
- kimi-k2.5
- deepseek-v4-pro
- gpt-oss:20b
- mistral-large-3:675b
- gemma3:27b
- glm-5
- deepseek-v4-flash
- gpt-oss:120b
- nemotron-3-nano:30b
- ministral-3:8b
- glm-5.1
- deepseek-v3.1:671b
- minimax-m2.1
- qwen3.5:397b
- nemotron-3-ultra
- glm-4.6
- minimax-m2.5
- ministral-3:3b
- rnj-1:8b
- glm-4.7
- qwen3-vl:235b-instruct
- minimax-m2
- minimax-m2.7
- devstral-small-2:24b
- deepseek-v3.2
- gemma4:31b
- nemotron-3-super
Model aktif sekarang: glm-5
Model tersedia:
- gpt-oss:120b (default)
- kimi-k2.6
- qwen3-next:80b
- qwen3-coder-next
- qwen3-vl:235b
- ministral-3:14b
- devstral-2:123b
- gemini-3-flash-preview
- gemma3:4b
- cogito-2.1:671b
- kimi-k2:1t
- kimi-k2-thinking
- qwen3-coder:480b
- minimax-m3
- gemma3:1

### Soal 2

**a. Apa yang dimaksud dengan Artificial Intelligence (AI)? Sebutkan dua contohnya dalam kehidupan sehari-hari.**

Artificial Intelligence (AI) adalah bidang ilmu komputer yang membuat mesin mampu meniru kemampuan cerdas manusia, seperti mengenali pola, memahami bahasa, mengambil keputusan, dan belajar dari data.

Dua contoh AI dalam kehidupan sehari-hari:

- Rekomendasi video di YouTube atau Netflix.
- Asisten virtual seperti Siri, Google Assistant, atau ChatGPT.

**b. Apa perbedaan antara Supervised Learning dan Unsupervised Learning? Berikan satu contoh untuk masing-masing.**

- **Supervised Learning** menggunakan data yang sudah memiliki label jawaban. Model belajar dari pasangan input-output yang benar. Contoh: klasifikasi email spam dan bukan spam.
- **Unsupervised Learning** menggunakan data tanpa label. Model mencari pola atau struktur sendiri. Contoh: clustering pelanggan berdasarkan perilaku belanja.

### Soal 3: Pertanyaan Konsep

**a. Apa itu Feature dalam konteks machine learning? Mengapa penting untuk memilih fitur yang tepat saat membangun model?**

Feature adalah variabel atau atribut yang digunakan model sebagai masukan untuk mempelajari pola. Contohnya umur, pendapatan, atau jumlah klik.

Pemilihan fitur yang tepat penting karena fitur yang relevan membantu model belajar lebih akurat, lebih cepat, dan lebih stabil. Fitur yang kurang tepat dapat membuat model sulit belajar atau menghasilkan prediksi yang kurang baik.

**b. Apa itu Fine-tuning dalam machine learning? Sebutkan satu kasus di mana fine-tuning berguna.**

Fine-tuning adalah proses menyesuaikan model yang sudah pre-trained agar lebih cocok dengan tugas atau data yang lebih spesifik.

Contoh kasus yang berguna: model bahasa umum di-fine-tune untuk klasifikasi sentimen ulasan pelanggan pada domain e-commerce atau layanan keuangan.
